# HDF5 to fbin

Convert an ANN-benchmarks `.hdf5` dataset into the `.fbin` format used by ParlayANN.

An `.fbin` file is a little-endian int32 `n`, an int32 `dims`, then `n * dims` row-major float32 values.
The `train` and `test` datasets are written to separate files.

In [1]:
from pathlib import Path

import h5py
import numpy as np

In [2]:
def write_fbin(vectors, path):
    """Write a 2D array to `path` in fbin format."""
    vectors = np.ascontiguousarray(vectors, dtype=np.float32)
    n, dims = vectors.shape
    with open(path, "wb") as f:
        np.array([n, dims], dtype=np.int32).tofile(f)
        vectors.tofile(f)
    print(f"wrote {path} ({n} points, dimension {dims})")

In [3]:
def hdf5_to_fbin(hdf5_path, out_dir=None):
    """Convert the train and test datasets of an hdf5 file into separate fbin files."""
    hdf5_path = Path(hdf5_path)
    out_dir = Path(out_dir) if out_dir else hdf5_path.parent / hdf5_path.stem
    out_dir.mkdir(parents=True, exist_ok=True)

    with h5py.File(hdf5_path, "r") as f:
        write_fbin(f["train"][:], out_dir / "base.fbin")
        write_fbin(f["test"][:], out_dir / "query.fbin")

    return out_dir

In [6]:
import os
os.listdir("../../Datasets/")

['mnist-784-euclidean.hdf5',
 'fashion_mnist-784-euclidean.hdf5',
 '.DS_Store',
 'fashion_mnist-784-euclidean',
 'glove25-25-angular.hdf5',
 'coco_i2i-512-angular.hdf5']

In [11]:
HDF5_PATH = "../../Datasets/mnist-784-euclidean.hdf5"

out_dir = hdf5_to_fbin(HDF5_PATH)
out_dir

wrote ../../Datasets/mnist-784-euclidean/base.fbin (60000 points, dimension 784)
wrote ../../Datasets/mnist-784-euclidean/query.fbin (10000 points, dimension 784)


PosixPath('../../Datasets/mnist-784-euclidean')

## Check the output

Read the headers back and confirm the first vector round-trips.

In [12]:
def read_fbin(path):
    """Read an fbin file back into a 2D array."""
    with open(path, "rb") as f:
        n, dims = np.fromfile(f, dtype=np.int32, count=2)
        return np.fromfile(f, dtype=np.float32).reshape(n, dims)


with h5py.File(HDF5_PATH, "r") as f:
    for name, fbin in [("train", "base.fbin"), ("test", "query.fbin")]:
        original, restored = f[name][:], read_fbin(out_dir / fbin)
        assert restored.shape == original.shape
        assert np.array_equal(restored, original.astype(np.float32))
        print(f"{name}: {restored.shape} matches")

train: (60000, 784) matches
test: (10000, 784) matches
